# 32 孟加拉国 DIA 分层身份揭示队列冻结（正式版）

本 Notebook 在读取任何化合物名称、分子式、结构、谱库命中或原论文身份之前，依据 Notebook 31 已冻结的跨极性簇排名固定后续身份揭示队列。

四个互斥队列均以跨极性簇为抽样单位，每组 100 个：

- `TOP_PRIORITY`：总排名前 100；
- `MID_PRIORITY`：总排名中央 10% 中确定性伪随机抽取 100；
- `LOW_PRIORITY`：总排名末尾 10% 中确定性伪随机抽取 100；
- `HIGH_RISK_HIGH_UNCERTAINTY`：风险分量和平均不确定性均不低于各自第 80 百分位，再按不确定性降序取 100。

固定种子为 `20260920`。输出记录输入/输出 SHA-256；身份揭示结果不得反向改变队列或 Notebook 31 排名。

> 本阶段只冻结验证设计，不进行谱库检索或 SIRIUS 注释。内核必须使用 `env_ms`。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'pbt_msms').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
assert Path(sys.prefix).name.lower() == 'env_ms', f'请切换到 env_ms 内核；当前为 {sys.executable}'
print('ROOT =', ROOT)
print('Python =', sys.executable)


In [ ]:
from pbt_msms.bangladesh_dia_external import paths
p = paths(ROOT)
RANK_DIR = p['case'] / '06_revealed_evaluation' / '31_feature_prioritization_v2'
rank_manifest = json.loads((RANK_DIR / '31_run_manifest.json').read_text(encoding='utf-8'))
assert rank_manifest['status'] == 'FEATURE_PRIORITIZATION_FROZEN'
assert rank_manifest['n_features'] == 18161
assert rank_manifest['n_cross_polarity_clusters'] == 16768
print('Notebook 31 frozen gate: PASS')


In [ ]:
from pbt_msms.bangladesh_dia_external import run_identity_reveal_cohort
manifest = run_identity_reveal_cohort(
    ROOT,
    n_per_cohort=100,
    seed=20260920,
    middle_bounds=(0.45, 0.55),
    low_bounds=(0.90, 1.00),
    high_risk_quantile=0.80,
    high_uncertainty_quantile=0.80,
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status'] == 'IDENTITY_REVEAL_COHORT_FROZEN'
assert manifest['n_clusters_selected'] == 400


In [ ]:
import pandas as pd
from pbt_msms.bangladesh_dia_external import sha256_file
OUT = p['case'] / '06_revealed_evaluation' / '32_identity_reveal_cohort_v2'
clusters = pd.read_csv(OUT / '32_frozen_cluster_cohort.csv')
members = pd.read_csv(OUT / '32_frozen_feature_members.csv.gz')
audit = pd.read_csv(OUT / '32_cohort_audit.csv')
assert len(clusters) == 400 and clusters['cross_polarity_cluster'].is_unique
assert clusters.groupby('cohort').size().eq(100).all()
assert set(members['cross_polarity_cluster']) == set(clusters['cross_polarity_cluster'])
assert not members.duplicated('spectrum_id').any()
for filename, expected_hash in manifest['output_hashes'].items():
    assert sha256_file(OUT / filename) == expected_hash, f'哈希不一致：{filename}'
display(audit)
display(clusters[['cohort', 'selection_order', 'cluster_rank', 'cross_polarity_cluster', 'modes', 'cluster_member_n', 'priority_score_primary', 'hazard_component', 'cluster_uncertainty_mean']].groupby('cohort').head(5))
print('四组互斥、成员完整、文件哈希：全部 PASS')


## 本阶段完成判据

最后一个单元格必须显示：`四组互斥、成员完整、文件哈希：全部 PASS`。

冻结目录：`external_data/bangladesh_surface_water/06_revealed_evaluation/32_identity_reveal_cohort_v2`。

下一阶段由 Notebook 33 对这 400 个冻结簇的全部谱图成员进行公开实测谱库匹配；随后 Notebook 34 不以谱库匹配状态作筛选，而是对全部 400 个冻结簇统一进行 SIRIUS 补充注释。